Dataset: pd_speech_features.csv (Sakar et al. UCI PD Speech dataset)
756 rows = 252 subjects x 3 recordings each, 752 acoustic features (baseline,
intensity, formants, MFCC, wavelet, TQWT) + gender + class.
 
CRITICAL: subjects have 3 recordings each. A plain stratified train_test_split
leaks subject identity across train/test (same person's voice in both splits),
which inflates F1/AUC and does not reflect real unseen-patient performance.
We use GroupShuffleSplit on `id` instead — same pattern as the Oxford sub-model.

In [11]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
 
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.frozen import FrozenEstimator
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_curve, confusion_matrix,
)

In [2]:
RANDOM_STATE = 42
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv(r"datasets\speech\parkinson+s+disease+classification\pd_speech_features.csv", skiprows=1)
df.head()

,id,gender,PPE,DFA,RPDE,numPulses,numPeriodsPulses,meanPeriodPulses,stdDevPeriodPulses,locPctJitter,...,tqwt_kurtosisValue_dec_28,tqwt_kurtosisValue_dec_29,tqwt_kurtosisValue_dec_30,tqwt_kurtosisValue_dec_31,tqwt_kurtosisValue_dec_32,tqwt_kurtosisValue_dec_33,tqwt_kurtosisValue_dec_34,tqwt_kurtosisValue_dec_35,tqwt_kurtosisValue_dec_36,class
0,0,1,0.85247,0.71826,0.57227,240,239,0.008064,0.000087,0.00218,...,1.5620,2.6445,3.8686,4.2105,5.1221,4.4625,2.6202,3.0004,18.9405,1
1,0,1,0.76686,0.69481,0.53966,234,233,0.008258,0.000073,0.00195,...,1.5589,3.6107,23.5155,14.1962,11.0261,9.5082,6.5245,6.3431,45.1780,1
2,0,1,0.85083,0.67604,0.58982,232,231,0.008340,0.000060,0.00176,...,1.5643,2.3308,9.4959,10.7458,11.0177,4.8066,2.9199,3.1495,4.7666,1
3,1,0,0.41121,0.79672,0.59257,178,177,0.010858,0.000183,0.00419,...,3.7805,3.5664,5.2558,14.0403,4.2235,4.6857,4.8460,6.2650,4.0603,1
4,1,0,0.32790,0.79782,0.53028,236,235,0.008162,0.002669,0.00535,...,6.1727,5.8416,6.0805,5.7621,7.7817,11.6891,8.2103,5.0559,6.1164,1


In [4]:
feature_cols = [c for c in df.columns if c not in ("id", "gender", "class")]
X = df[feature_cols].values.astype(np.float64)
y = df["class"].values.astype(int)
groups = df["id"].values  # subject id - 252 unique subjects, 3 rows each
 
print(f"Rows: {X.shape[0]}, Features: {X.shape[1]}, Subjects: {len(set(groups))}")
print(f"Class balance: {np.bincount(y)}")

Rows: 756, Features: 752, Subjects: 252
Class balance: [192 564]


In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
 
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train = groups[train_idx]
 
# Sanity check: no subject appears in both splits
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx])), "LEAKAGE: subject in both splits"
print(f"Train: {len(train_idx)} rows / {len(set(groups_train))} subjects")
print(f"Test:  {len(test_idx)} rows / {len(set(groups[test_idx]))} subjects")

Train: 603 rows / 201 subjects
Test:  153 rows / 51 subjects


In [12]:
K_CANDIDATES = [30, 50, 75, 100, 150, 200]
gkf_sweep = GroupKFold(n_splits=5)
 
best_k, best_k_f1 = None, -1.0
for k in K_CANDIDATES:
    fold_f1s = []
    for tr_i, va_i in gkf_sweep.split(X_train, y_train, groups=groups_train):
        sel = SelectKBest(f_classif, k=k)
        Xtr_k = sel.fit_transform(X_train[tr_i], y_train[tr_i])
        Xva_k = sel.transform(X_train[va_i])
 
        m = RandomForestClassifier(
            n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
        )
        m.fit(Xtr_k, y_train[tr_i])
        pred = m.predict(Xva_k)
        fold_f1s.append(f1_score(y_train[va_i], pred))
 
    mean_f1 = float(np.mean(fold_f1s))
    print(f"  k={k:4d}  grouped CV F1 = {mean_f1:.4f} (+/- {np.std(fold_f1s):.4f})")
    if mean_f1 > best_k_f1:
        best_k, best_k_f1 = k, mean_f1
 
print(f"\nBest k_features by grouped CV: {best_k} (F1={best_k_f1:.4f})")
K_FEATURES = best_k
 
selector = SelectKBest(f_classif, k=K_FEATURES)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)
 
selected_idx = selector.get_support(indices=True)
selected_feature_names = [feature_cols[i] for i in selected_idx]
print(f"Selected {len(selected_feature_names)} features via ANOVA F-test")

  k=  30  grouped CV F1 = 0.8685 (+/- 0.0438)
  k=  50  grouped CV F1 = 0.8567 (+/- 0.0438)
  k=  75  grouped CV F1 = 0.8728 (+/- 0.0387)
  k= 100  grouped CV F1 = 0.8682 (+/- 0.0336)
  k= 150  grouped CV F1 = 0.8732 (+/- 0.0401)
  k= 200  grouped CV F1 = 0.8805 (+/- 0.0444)

Best k_features by grouped CV: 200 (F1=0.8805)
Selected 200 features via ANOVA F-test


In [13]:
gkf = GroupKFold(n_splits=5)
cv_f1 = []
for tr_i, va_i in gkf.split(X_train_sel, y_train, groups=groups_train):
    m = RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    )
    m.fit(X_train_sel[tr_i], y_train[tr_i])
    pred = m.predict(X_train_sel[va_i])
    cv_f1.append(f1_score(y_train[va_i], pred))
 
print(f"\nGrouped 5-fold CV F1 @ threshold=0.5, k={K_FEATURES}: {np.mean(cv_f1):.4f} (+/- {np.std(cv_f1):.4f})")
decision_threshold = 0.5


Grouped 5-fold CV F1 @ threshold=0.5, k=200: 0.8887 (+/- 0.0307)


In [14]:
base_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
base_model.fit(X_train_sel, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [15]:
cal_model = CalibratedClassifierCV(
    estimator=FrozenEstimator(base_model),
    method="sigmoid",
)
cal_model.fit(X_train_sel, y_train)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",FrozenEstimat...dom_state=42))
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gin

In [16]:
test_probs = cal_model.predict_proba(X_test_sel)[:, 1]
 
preds_default = (test_probs >= 0.5).astype(int)
preds_tuned = (test_probs >= decision_threshold).astype(int)
 
val_auc = roc_auc_score(y_test, test_probs)  # threshold-independent
 
print("\n========== TEST RESULTS @ threshold=0.5 (for comparison) ==========")
print("Accuracy :", round(accuracy_score(y_test, preds_default), 4))
print("Precision:", round(precision_score(y_test, preds_default), 4))
print("Recall   :", round(recall_score(y_test, preds_default), 4))
print("F1 Score :", round(f1_score(y_test, preds_default), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, preds_default))
 
print(f"\n========== TEST RESULTS @ CV-tuned threshold={decision_threshold:.4f} ==========")
print("ROC-AUC  :", round(val_auc, 4), " (threshold-independent, same either way)")
print("Accuracy :", round(accuracy_score(y_test, preds_tuned), 4))
print("Precision:", round(precision_score(y_test, preds_tuned), 4))
print("Recall   :", round(recall_score(y_test, preds_tuned), 4))
print("F1 Score :", round(f1_score(y_test, preds_tuned), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, preds_tuned))


========== TEST RESULTS @ threshold=0.5 (for comparison) ==========
Accuracy : 0.7712
Precision: 0.7836
Recall   : 0.9459
F1 Score : 0.8571
Confusion matrix:
 [[ 13  29]
 [  6 105]]

========== TEST RESULTS @ CV-tuned threshold=0.5000 ==========
ROC-AUC  : 0.9204  (threshold-independent, same either way)
Accuracy : 0.7712
Precision: 0.7836
Recall   : 0.9459
F1 Score : 0.8571
Confusion matrix:
 [[ 13  29]
 [  6 105]]


In [17]:
N_BOOTSTRAP = 50
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_models = []
n_train = X_train_sel.shape[0]
 
for i in range(N_BOOTSTRAP):
    boot_idx = rng.integers(0, n_train, size=n_train)
    m = RandomForestClassifier(
        n_estimators=100, max_depth=15, min_samples_split=5, min_samples_leaf=2,
        class_weight="balanced", random_state=RANDOM_STATE + i, n_jobs=-1,
    )
    m.fit(X_train_sel[boot_idx], y_train[boot_idx])
    bootstrap_models.append(m)
 
print(f"\nTrained {N_BOOTSTRAP} bootstrap models for MC uncertainty estimation")


Trained 50 bootstrap models for MC uncertainty estimation


In [18]:
joblib.dump(base_model, ARTIFACT_DIR / "speech_uci_rf_v1.pkl")
joblib.dump(cal_model, ARTIFACT_DIR / "speech_uci_rf_v1_calibrated.pkl")
joblib.dump(selector, ARTIFACT_DIR / "feature_selector.pkl")
joblib.dump(feature_cols, ARTIFACT_DIR / "feature_cols_full.pkl")
joblib.dump(selected_feature_names, ARTIFACT_DIR / "selected_feature_names.pkl")
joblib.dump(bootstrap_models, ARTIFACT_DIR / "bootstrap_models.pkl")
joblib.dump(val_auc, ARTIFACT_DIR / "validation_auc.pkl")
joblib.dump(decision_threshold, ARTIFACT_DIR / "decision_threshold.pkl")
 
print("\nSaved artifacts:")
for f in sorted(ARTIFACT_DIR.glob("*.pkl")):
    print(" -", f.name)
print(f"\nValidation AUC (IntraModalFuser weight): {val_auc:.4f}")
print(f"Decision threshold (hard-label cutoff):   {decision_threshold:.4f}")


Saved artifacts:
 - bootstrap_models.pkl
 - decision_threshold.pkl
 - feature_cols_full.pkl
 - feature_selector.pkl
 - selected_feature_names.pkl
 - speech_uci_rf_v1.pkl
 - speech_uci_rf_v1_calibrated.pkl
 - validation_auc.pkl

Validation AUC (IntraModalFuser weight): 0.9204
Decision threshold (hard-label cutoff):   0.5000
